In [3]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict, List, Optional, Literal
import subprocess
from openai import OpenAI
import json
import os
import re
import requests
from bs4 import BeautifulSoup


class Question(TypedDict):
    id: str
    text: str
    choices: List[str]   # ["A) ...", "B) ...", ...]
    correct: str         # "A" ~ "E"
    year: int
    grade: int

class GaussState(TypedDict):
    grade: int
    problem_bank: List[Question]
    current_question: Optional[Question]
    answered_count: int
    correct_count: int
    score: int
    last_answer: Optional[str]

#### Tools

#### Web Parsing Functions
def _parse_solutions(solution_url: str, grade: int, year: int):
    """Get the answers from the Solution HTML"""
    try:
        response = requests.get(solution_url, timeout=15)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')
        
        all_text = soup.get_text()
        grade_marker = f"Grade {grade}"  # 올바른 마커
        
        if grade_marker not in all_text:
            print(f"Warning! Can't find the Solution for Grade {grade}")
            return {}
        
        grade_start = all_text.find(grade_marker)
        grade_text = all_text[grade_start:]
        
        next_grade = re.search(r"Grade \d+", grade_text[len(grade_marker):])  # 올바른 정규식
        if next_grade:
            grade_text = grade_text[:len(grade_marker) + next_grade.start()]
        
        pattern = r'Answer:\s*\(([A-E])\)'
        answers = re.findall(pattern, grade_text)
        
        answers_dict = {}
        for idx, answer in enumerate(answers, 1):
            answers_dict[idx] = answer
        
        print(f"  ✓ Loaded {len(answers_dict)} answers")
        return answers_dict
    
    except Exception as e:
        print(f"  ✗ Failed to parse answers: {e}")
        return {}
    
def _parse_from_web(grade: int, year: int) -> List[Question]:
    """CEMC Gauss Problems and Answers for the GRADE and YEAR """
    
    # URL 구성
    problem_url = f"https://cemc.uwaterloo.ca/sites/default/files/documents/{year}/{year}Gauss{grade}Contest.html"
    solution_url = f"https://cemc.uwaterloo.ca/sites/default/files/documents/{year}/{year}GaussSolution.html"
    
    try:
        print(f"Loading... {year} Grade {grade} answers...")
        answers_dict = _parse_solutions(solution_url, grade, year)
        
        print(f"Loading... {year} Grade {grade} problems...")
        response = requests.get(problem_url, timeout=15)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')
        
        questions = []
        all_text = soup.get_text()
        
        pattern = r'(\d+)\.\s+([^\n]+(?:\n(?!(?:\d+\.|Answer:))[^\n]*)*)'
        matches = re.finditer(pattern, all_text)
        
        question_count = 0
        for match in matches:
            question_num = int(match.group(1))
            question_text = match.group(2).strip()
            
            choices = re.findall(r'\(([A-E])\)\s*([^\n]+)', question_text)
            
            if len(choices) >= 5:
                formatted_choices = [f"{letter}) {text.strip()}" for letter, text in choices[:5]]
                problem_only = re.sub(r'\([A-E]\)\s*[^\n]+', '', question_text).strip()
                correct_answer = answers_dict.get(question_num, "A")
                
                question_obj: Question = {
                    "id": f"{year}-G{grade}-Q{question_num}",
                    "text": problem_only,
                    "choices": formatted_choices,
                    "correct": correct_answer,
                    "year": year,
                    "grade": grade,
                }
                questions.append(question_obj)
                question_count += 1
        
        print(f"  ✓ Loaded total {question_count} prombles")
        return questions
    
    except Exception as e:
        print(f"  ✗ Failed to load problems: {e}")
        return []


# Step 1: One time parsing and saving Json for the year
def cache_problems_from_web(grade: int, year: int):
    """ Save Json files for each year's problems parsing from the web"""
    problems = load_problems_from_past_contests(grade, year)
    cache_file = f"gauss_problems_grade{grade}_{year}.json"
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(problems, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved {cache_file} json file \n")


# Caching and parsing for all years of the grade
def cache_all_years(grade: int, years: List[int]):
    """Cashing for all years"""
    print(f"\n🔄 Begin Grade {grade} for all years data...\n")
    
    for year in years:
        print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        print(f"Year: {year}")
        print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
        cache_problems_from_web(grade, year)
    
    print(f"✅ Completed to cashe Grade {grade} !")


# Step 2: Load only from JSON after
def load_problems_from_past_contests(grade: int, year: int) -> List[Question]:
    """년도별 문제 로드 (캐시 우선, 미존재시 웹 파싱)"""
    cache_file = f"gauss_problems_grade{grade}_{year}.json"
    
    if os.path.exists(cache_file):
        print(f"[캐시] {cache_file}에서 로드 중...")
        with open(cache_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            print(f"✓ {len(data)}개 문제 로드됨\n")
            return data
    
    return _parse_from_web(grade, year)


# 특정 학년의 전체 년도 데이터를 합쳐서 로드
def load_problems_all_years(grade: int, years: List[int]) -> List[Question]:
    """여러 년도의 문제를 모두 로드해서 합치기"""
    all_problems = []
    
    for year in years:
        problems = load_problems_from_past_contests(grade, year)
        all_problems.extend(problems)
    
    return all_problems


#### 테스트: 샘플 데이터 확인
print("=" * 70)
print("🚀 Gauss 문제 로딩 테스트 (년도별)")
print("=" * 70 + "\n")

# 방법 1: 단일 년도 로드
print("📌 방법 1: 2023년 Grade 7 문제 로드\n")
problems_2023 = load_problems_from_past_contests(7, 2023)

print("📋 샘플 문제 (2023년):\n")
for i, problem in enumerate(problems_2023[:2], 1):
    print(f"문제 {i}: {problem['id']}")
    print(f"  텍스트: {problem['text'][:60]}...")
    print(f"  정답: {problem['correct']}")
    print()

# 방법 2: 여러 년도 한 번에 캐시 (첫 실행용)
# 이 부분은 시간이 걸리므로 필요시만 실행
# cache_all_years(7, [2023, 2024, 2025])

# 방법 3: 캐시된 데이터 여러 년도 로드
print("━" * 70)
print("\n📌 방법 2: 2024년 Grade 7 문제 로드\n")
problems_2024 = load_problems_from_past_contests(7, 2024)

print("📋 샘플 문제 (2024년):\n")
for i, problem in enumerate(problems_2024[:1], 1):
    print(f"문제 {i}: {problem['id']}")
    print(f"  텍스트: {problem['text'][:60]}...")
    print(f"  정답: {problem['correct']}")
    print()

# def load_problems_from_past_contests(grade: int) -> List[Question]:
#     """
#     TODO:
#       - CEMC Gauss Past Contests PDF를 파싱해서 JSON/DB로 저장해두고 여기서 로드.
#       - 지금은 간단한 예시 dummy 데이터.
#     """
#     dummy_questions: List[Question] = [
#         {
#             "id": "2024-G7-Q1",
#             "text": "What is 2 + 3?",
#             "choices": ["A) 4", "B) 5", "C) 6", "D) 7", "E) 8"],
#             "correct": "B",
#             "year": 2024,
#             "grade": grade,
#         },
#         {
#             "id": "2024-G7-Q2",
#             "text": "What is 10 - 4?",
#             "choices": ["A) 5", "B) 6", "C) 7", "D) 8", "E) 9"],
#             "correct": "B",
#             "year": 2024,
#             "grade": grade,
#         },
#     ]
#     return dummy_questions




🚀 Gauss 문제 로딩 테스트 (년도별)

📌 방법 1: 2023년 Grade 7 문제 로드

Loading... 2023 Grade 7 answers...
  ✓ Loaded 0 answers
Loading... 2023 Grade 7 problems...
  ✓ Loaded total 0 prombles
📋 샘플 문제 (2023년):

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📌 방법 2: 2024년 Grade 7 문제 로드

Loading... 2024 Grade 7 answers...
  ✓ Loaded 0 answers
Loading... 2024 Grade 7 problems...
  ✓ Loaded total 0 prombles
📋 샘플 문제 (2024년):

